# Lab 10 — PID digital: discretização, filtro derivativo e anti-windup

**Unidade IV — Projeto, sintonia e implementação de PID** · conteúdos 4.1/4.2 do PPC

**Objetivos:**
1. Discretizar o PID e estudar o efeito do período de amostragem $h$;
2. Escrever o PID **como será embarcado** (função passo a passo, pronta para Arduino/microcontrolador);
3. Integrar todas as proteções: derivada sobre a medição, filtro, saturação e anti-windup;
4. Validar por simulação híbrida (planta contínua + controlador discreto).

**Referências:** Åström & Hägglund, *Advanced PID Control*, cap. 13 · Åström & Murray (FBS), cap. 11.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import control as ct
    print("python-control", ct.__version__)
except ImportError:
    %pip install control
    import control as ct

## 1. Do contínuo ao discreto com `sample_system`

Planta e sintonia de referência (Lab 09, planta B com PI-lambda):

In [ ]:
K_B, tau_B, theta_B = 3.0, 4.0, 1.5
num_p, den_p = ct.pade(theta_B, 5)
G = ct.tf([K_B], [tau_B, 1]) * ct.tf(num_p, den_p)

# sintonia lambda conservadora
lam = tau_B
Kp = tau_B / (K_B * (lam + theta_B))
Ti = tau_B
Td = 0.0
print(f"Sintonia PI: Kp = {Kp:.3f}, Ti = {Ti:.1f} s")

C_cont = ct.tf([Kp * Ti, Kp], [Ti, 0])   # PI contínuo

# discretização por Tustin (transformação bilinear) para vários h
for h in [0.1, 0.5, 2.0]:
    C_d = ct.sample_system(C_cont, h, method='tustin')
    print(f"h = {h} s -> C(z) = {C_d}")

## 2. Quanto custa amostrar devagar?

O segurador de ordem zero atrasa em média $h/2$, custando fase $\Delta\phi \approx \omega h/2$.
Simulação híbrida: planta contínua, controlador discreto.

In [ ]:
def simulate_hybrid(C_d, h, t_end=40.0, r_val=1.0, dist=0.0, t_dist=25.0):
    """Simula planta contínua G com controlador discreto C_d (período h)."""
    n_steps = int(t_end / h)
    sysd_ctrl = ct.ss(C_d)                     # controlador em espaço de estados discreto
    sys_plant = ct.ss(G)                       # planta contínua

    xc = np.zeros(sysd_ctrl.nstates)           # estado do controlador
    xp = np.zeros(sys_plant.nstates)           # estado da planta
    t_hist, y_hist, u_hist = [], [], []
    y = 0.0
    for k in range(n_steps):
        t_k = k * h
        r = r_val
        e = r - y
        # passo do controlador discreto
        u = float(sysd_ctrl.C @ xc + sysd_ctrl.D * e)
        xc = sysd_ctrl.A @ xc + (sysd_ctrl.B * e).flatten()
        # perturbação na entrada da planta
        u_planta = u + (dist if t_k >= t_dist else 0.0)
        # integra a planta contínua durante o período h (ZOH)
        t_seg = np.linspace(0, h, 5)
        resp = ct.forced_response(sys_plant, t_seg,
                                  u_planta * np.ones_like(t_seg), X0=xp)
        xp = resp.states[:, -1]
        y = float(resp.outputs[-1])
        t_hist.append(t_k); y_hist.append(y); u_hist.append(u)
    return np.array(t_hist), np.array(y_hist), np.array(u_hist)

plt.figure(figsize=(9, 5))
# referência contínua ideal
T_ideal = ct.feedback(C_cont * G, 1)
resp_i = ct.step_response(T_ideal, np.linspace(0, 40, 2000))
plt.plot(resp_i.time, resp_i.outputs, 'k--', lw=2, label='contínuo ideal')

for h in [0.1, 1.0, 3.0]:
    C_d = ct.sample_system(C_cont, h, method='tustin')
    t_h, y_h, _ = simulate_hybrid(C_d, h)
    plt.step(t_h, y_h, where='post', lw=1.5, label=f'digital, h = {h} s')

plt.axhline(1, color='gray', ls=':')
plt.xlabel('Tempo [s]'); plt.ylabel('y(t)')
plt.title('Efeito do período de amostragem (regra: h ≤ τ/10)')
plt.legend(); plt.grid(True)
plt.show()

Com $h = 0{,}1$ s (τ/40) o digital é indistinguível do contínuo; com $h = 3$ s (≈ 0,75τ) a perda
de fase degrada visivelmente o amortecimento. **Regra do curso: $h \le \tau/10$ (ou $T_u/10$).**

## 3. O PID de produção: código embarcável

A função abaixo é o **produto final da disciplina** — a mesma estrutura será gravada no
microcontrolador do projeto final. Inclui: forma ISA, ponderação de referência $b$, derivada
**sobre a medição** com filtro, saturação e anti-windup por back-calculation.

In [ ]:
class DigitalPID:
    """PID digital ISA com filtro derivativo, ponderacao de referencia e anti-windup.

    Implementa, a cada chamada de step():
        P_k = Kp (b r_k - y_k)
        I_k = I_{k-1} + Kp h/Ti e_k + h/Tt (u_sat - u)
        D_k = a_d D_{k-1} - b_d (y_k - y_{k-1})
        u_k = sat(P_k + I_k + D_k)
    """

    def __init__(self, Kp, Ti=np.inf, Td=0.0, h=0.1, N=10, b=1.0,
                 umin=-np.inf, umax=np.inf, Tt=None):
        self.Kp, self.Ti, self.Td = Kp, Ti, Td
        self.h, self.N, self.b = h, N, b
        self.umin, self.umax = umin, umax
        # constante de anti-windup: padrão sqrt(Ti Td) ou Ti/2
        if Tt is None:
            Tt = np.sqrt(Ti * Td) if (np.isfinite(Ti) and Td > 0) else \
                 (Ti / 2 if np.isfinite(Ti) else np.inf)
        self.Tt = Tt
        # coeficientes pre-calculados do filtro derivativo
        if Td > 0:
            self.ad = Td / (Td + N * h)
            self.bd = Kp * Td * N / (Td + N * h)
        else:
            self.ad = self.bd = 0.0
        self.reset()

    def reset(self):
        self.I = 0.0        # parcela integral acumulada
        self.D = 0.0        # parcela derivativa filtrada
        self.y_prev = None  # medicao anterior

    def step(self, r, y):
        """Executa um periodo de amostragem; retorna o controle u aplicavel."""
        if self.y_prev is None:
            self.y_prev = y   # inicializacao sem salto derivativo

        e = r - y
        P = self.Kp * (self.b * r - y)                     # proporcional ponderado
        self.D = self.ad * self.D - self.bd * (y - self.y_prev)  # derivada da MEDICAO
        v = P + self.I + self.D                            # controle pedido
        u = np.clip(v, self.umin, self.umax)               # controle aplicado

        # atualizacao do integrador com back-calculation
        if np.isfinite(self.Ti):
            self.I += self.Kp * self.h / self.Ti * e
            if np.isfinite(self.Tt):
                self.I += self.h / self.Tt * (u - v)

        self.y_prev = y
        return u

## 4. Validação: PID digital completo com saturação

Planta com atuador limitado a $u \in [0;\ 0{,}5]$ e degrau de referência que força a saturação.

In [ ]:
def run_pid(pid, t_end=60.0, r_val=1.5, dist=-0.3, t_dist=40.0):
    """Malha: DigitalPID -> saturacao (interna ao PID) -> planta continua."""
    sys_plant = ct.ss(G)
    xp = np.zeros(sys_plant.nstates)
    h = pid.h
    n = int(t_end / h)
    t_hist, y_hist, u_hist, i_hist = [], [], [], []
    y = 0.0
    for k in range(n):
        t_k = k * h
        u = pid.step(r_val, y)
        u_tot = u + (dist if t_k >= t_dist else 0.0)
        t_seg = np.linspace(0, h, 5)
        resp = ct.forced_response(sys_plant, t_seg,
                                  u_tot * np.ones_like(t_seg), X0=xp)
        xp = resp.states[:, -1]
        y = float(resp.outputs[-1])
        t_hist.append(t_k); y_hist.append(y); u_hist.append(u); i_hist.append(pid.I)
    return map(np.array, (t_hist, y_hist, u_hist, i_hist))

h = 0.1
pid_sem_aw = DigitalPID(Kp, Ti, h=h, umin=0, umax=0.5, Tt=np.inf)   # anti-windup desligado
pid_com_aw = DigitalPID(Kp, Ti, h=h, umin=0, umax=0.5)              # anti-windup padrao

t1, y1, u1, i1 = run_pid(pid_sem_aw)
t2, y2, u2, i2 = run_pid(pid_com_aw)

fig, axs = plt.subplots(3, 1, figsize=(9, 9), sharex=True)
axs[0].plot(t1, y1, 'C3', lw=2, label='sem anti-windup')
axs[0].plot(t2, y2, 'C2', lw=2, label='com anti-windup')
axs[0].axhline(1.5, color='gray', ls='--')
axs[0].set_ylabel('y'); axs[0].legend(); axs[0].grid(True)

axs[1].step(t1, u1, 'C3', lw=1.5, where='post')
axs[1].step(t2, u2, 'C2', lw=1.5, where='post')
axs[1].axhline(0.5, color='gray', ls=':')
axs[1].set_ylabel('u (saturado em 0,5)'); axs[1].grid(True)

axs[2].plot(t1, i1, 'C3', lw=2)
axs[2].plot(t2, i2, 'C2', lw=2)
axs[2].set_ylabel('parcela I'); axs[2].set_xlabel('Tempo [s]'); axs[2].grid(True)
fig.suptitle('PID digital completo: saturação + perturbação em t = 40 s')
plt.show()

O controlador da classe `DigitalPID` reproduz em código embarcável tudo o que foi estudado:
a versão com anti-windup satura sem "explodir" o integrador e ainda rejeita a perturbação.

## 5. Ponte para o hardware (projeto final)

O laço de controle no Arduino/microcontrolador tem exatamente a estrutura de `run_pid`:

```c
// pseudocódigo do laço embarcado (período h garantido por timer)
void loop_controle() {
    y = ler_sensor();                  // ADC/encoder
    u = pid_step(r, y);                // mesma matemática da classe DigitalPID
    escrever_atuador(u);               // PWM
    registrar(t, r, y, u);             // telemetria via serial p/ análise no Python
}
```

A telemetria serial permite trazer os dados de volta ao notebook e comparar bancada × modelo.

---
> **🖼️ Figuras de apoio nos livros:**
> - Nise, **Figura 13.4** — conversão A/D: sinal analógico, amostrador + segurador (ZOH) e amostras digitais. Cap. 13, p. 1039 do arquivo PDF (a cópia digital não exibe o nº impresso).

## Exercícios (relatório do Lab 10)

**E1.** Compare as discretizações `tustin`, `zoh` e `euler` (forward) do PI para h = 1 s.
Alguma delas instabiliza a malha híbrida? (`ct.sample_system` aceita `method=`.)

**E2.** Adicione ruído de medição gaussiano ($\sigma = 0{,}01$) em `run_pid` e ative a derivada
($T_d = 0{,}75$). Compare a atividade do controle com N = 5 e N = 50.

**E3.** Meça o maior h que mantém $M_p \le 20\,\%$ nesta malha. Confronte com as regras
$h \le \tau/10$ e $h \le T_u/10$.

**E4.** Estenda `DigitalPID` com transferência *bumpless* manual/automático: um método
`set_manual(u_manual)` que, ao voltar para automático, inicialize `I` de modo que o controle
não salte. Demonstre em simulação.

In [ ]:
# E1 — sua solução aqui

In [ ]:
# E2 — sua solução aqui

In [ ]:
# E3 — sua solução aqui

In [ ]:
# E4 — sua solução aqui